# nb06 — Qwen3 Vector Store Index And Retrieval Evaluation

**Purpose.** nb05 selected `Qwen/Qwen3-Embedding-0.6B` as the strongest dense retrieval model for the Kalisio corpus. This notebook turns that decision into real vector-store indexes and checks whether persisted retrieval preserves the nb05 behavior.

The notebook builds a Qdrant collection first, then can reuse the same Qwen3 vectors to compare Chroma and LanceDB on the same chunks. It records index manifests, runs smoke queries, and replays the 200-question nb05 gold set against the real stores.

This closes the loop between the offline nb05 benchmark and the Phase 4 retrieval pipeline: same corpus, same chunks, same embeddings, different storage/search backends.


In [1]:
import gc, json, sys, time
from pathlib import Path

import pandas as pd

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "data").exists():
            return candidate
        knowledge = candidate / "knowledge"
        if (knowledge / "pyproject.toml").exists() and (knowledge / "data").exists():
            return knowledge
    raise FileNotFoundError("Could not find the knowledge project root")

# ── path setup for experiment_helper backup layout ──
def _find_repo_root():
    for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (p / "pyproject.toml").exists():
            return p
    raise RuntimeError("Cannot find knowledge repo root")

ROOT = _find_repo_root()
_HELPER = ROOT / "docs" / "experiments" / "experiment_helper"
_LAB = ROOT / "docs" / "experiments"
sys.path.insert(0, str(_HELPER))
sys.path.insert(0, str(_LAB / "chunking_lab"))
sys.path.insert(0, str(_LAB / "embedding_lab"))
sys.path.insert(0, str(_LAB / "retrieval_lab"))
sys.path.insert(0, str(_LAB / "shared"))

OUTPUT_DIR = ROOT / "outputs"
MANIFEST_PATH = OUTPUT_DIR / "nb06_qdrant_manifest.json"

pd.set_option("display.max_colwidth", 120)
print(f"[setup] root={ROOT}")

[setup] root=/home/felix/kalisio/knowledge


## Configuration

The defaults match the Phase 4 decision: Qwen3 embeddings and Qdrant as the reference vector store. Start Qdrant with `docker compose up -d qdrant` before running the indexing cell.

In [2]:
from nb06_helpers import (
    COLLECTION_NAME,
    CHROMA_COLLECTION_NAME,
    CHROMA_PATH,
    LANCEDB_PATH,
    LANCEDB_TABLE_NAME,
    QDRANT_URL,
    QWEN_MODEL_ID,
    IndexBuildConfig,
    VectorStoreConfig,
    build_qdrant_index,
    build_chroma_collection,
    build_lancedb_table,
    build_qdrant_index_from_records,
    chroma_file_rankings,
    evaluate_file_rankings,
    lancedb_file_rankings,
    create_qdrant_client,
    encode_queries,
    load_gold,
    load_qwen_model,
    qdrant_file_rankings,
    normalize_chunks,
    query_points,
    scan_and_chunk_corpus,
    vector_records_from_chunks,
    write_json,
)

config = IndexBuildConfig(
    collection_name=COLLECTION_NAME,
    qdrant_url=QDRANT_URL,
    model_id=QWEN_MODEL_ID,
    batch_size=4,
    recreate_collection=True,
    upsert_batch_size=64,
)

store_config = VectorStoreConfig()

config, store_config

(IndexBuildConfig(collection_name='kalisio_qwen3_nb06_v1', qdrant_url='http://localhost:6333', model_id='Qwen/Qwen3-Embedding-0.6B', batch_size=4, distance='cosine', recreate_collection=True, upsert_batch_size=64),
 VectorStoreConfig(chroma_path='vector_db/chroma_nb06', chroma_collection_name='kalisio_qwen3_nb06_v1', lancedb_path='vector_db/lancedb_nb06', lancedb_table_name='kalisio_qwen3_nb06_v1'))

## Corpus And Chunks

This uses the same corpus scope as nb05: the JS/Vue RAG profile, with `docs` and `tools` re-included, and with Markdown, JS, Vue, and selected JSON files indexed.

In [3]:
scan, chunks = scan_and_chunk_corpus()
records = normalize_chunks(chunks)

print(f"[corpus] files={len(scan.included)} chunks={len(chunks)} records={len(records)}")
pd.DataFrame([
    {
        "repository": r["payload"]["repository"],
        "file_type": r["payload"]["file_type"],
        "chunk_type": r["payload"]["chunk_type"],
        "source_path": r["payload"]["source_path"],
        "text_chars": r["payload"]["text_chars"],
    }
    for r in records[:10]
])

[corpus] files=1251 chunks=9893 records=9893


,repository,file_type,chunk_type,source_path,text_chars
0,crisis,md,markdown,crisis/README.md,141
1,crisis,md,markdown,crisis/README.md,1891
2,crisis,md,markdown,crisis/README.md,534
3,crisis,js,javascript,crisis/api/src/channels.js,817
4,crisis,js,javascript,crisis/api/src/channels.js,244
5,crisis,js,javascript,crisis/api/src/channels.js,776
6,crisis,js,javascript,crisis/api/src/channels.js,809
7,crisis,js,javascript,crisis/api/src/channels.js,325
8,crisis,js,javascript,crisis/api/src/hooks.js,404
9,crisis,js,javascript,crisis/api/src/hooks.js,663


In [4]:
summary = pd.DataFrame([
    {
        "repository": r["payload"]["repository"],
        "file_type": r["payload"]["file_type"],
        "chunk_type": r["payload"]["chunk_type"],
    }
    for r in records
])

print(summary.groupby("repository").size().sort_values(ascending=False).to_frame("chunks").to_string())
print(summary.groupby(["file_type", "chunk_type"]).size().sort_values(ascending=False).to_frame("chunks").to_string())

            chunks
repository        
kdk           6497
crisis        1930
kano           648
kapp           492
skeleton       326
                              chunks
file_type chunk_type                
js        javascript            3908
md        markdown              1846
vue       script                1733
json      i18n_translations      842
vue       template               681
mjs       javascript             595
json      schemas_validation     222
vue       style                   65
json      test_config              1


## Build The Qdrant Index

This is the heavy cell. It loads Qwen3, encodes every chunk, recreates the Qdrant collection, upserts points, and writes `outputs/nb06_qdrant_manifest.json`.

In [5]:
model = load_qwen_model()
t0 = time.perf_counter()
vector_records, corpus_vectors = vector_records_from_chunks(
    chunks,
    model,
    batch_size=config.batch_size,
)
manifest = build_qdrant_index_from_records(
    config=config,
    records=vector_records,
    vector_size=int(corpus_vectors.shape[1]),
    files_included=len(scan.included),
    chunks=chunks,
    manifest_path=MANIFEST_PATH,
)
elapsed = time.perf_counter() - t0

del model, corpus_vectors
gc.collect()

print(f"[index] qdrant built in {elapsed:.1f}s")
manifest


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

[encode] passages=9893 batch_size=4
[index] qdrant built in 117.2s


{'index_version': 'nb06.qdrant.v1',
 'collection_name': 'kalisio_qwen3_nb06_v1',
 'qdrant_url': 'http://localhost:6333',
 'model_id': 'Qwen/Qwen3-Embedding-0.6B',
 'distance': 'cosine',
 'vector_size': 1024,
 'files_included': 1251,
 'chunks': 9893,
 'points_count': 9893,
 'seconds': 9.251,
 'chunks_per_sec': 1069.393}

## Collection Smoke Check

Use a few developer-style questions to verify that Qdrant returns payloads with stable paths and readable text.

In [6]:
client = create_qdrant_client(config.qdrant_url)
info = client.get_collection(config.collection_name)
print(f"[qdrant] collection={config.collection_name} points={info.points_count}")

model = load_qwen_model()
questions = [
    "How do I add a new layer to a KDK map?",
    "Où est implémentée la logique de géolocalisation dans KDK ?",
    "How are hooks registered in a Kalisio application?",
]
query_vectors = encode_queries(model, questions)

rows = []
for question, vector in zip(questions, query_vectors):
    for rank, point in enumerate(query_points(client, config.collection_name, vector.tolist(), limit=5), start=1):
        payload = point.payload or {}
        rows.append({
            "question": question,
            "rank": rank,
            "score": round(point.score, 4),
            "source_path": payload.get("source_path"),
            "chunk_type": payload.get("chunk_type"),
            "preview": (payload.get("text") or "")[:180].replace("\n", " "),
        })

del model
gc.collect()
pd.DataFrame(rows)

[qdrant] collection=kalisio_qwen3_nb06_v1 points=9893


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

[encode] queries=3 batch_size=16


,question,rank,score,source_path,chunk_type,preview
0,How do I add a new layer to a KDK map?,1,0.7135,kdk/map/client/i18n/map_en.json,i18n_translations,"// kdk/map/client/i18n/map_en.json :: KAddLayer { ""KAddLayer"": { ""NAME_FIELD_LABEL"": ""Enter the name of the ne..."
1,How do I add a new layer to a KDK map?,2,0.6897,kdk/map/client/i18n/map_fr.json,i18n_translations,"// kdk/map/client/i18n/map_fr.json :: KAddLayer { ""KAddLayer"": { ""NAME_FIELD_LABEL"": ""Saisissez le nom de la n..."
2,How do I add a new layer to a KDK map?,3,0.6844,kdk/docs/api/map/map-mixins.md,markdown,Context: Map Mixins > Base Map Source: kdk/docs/api/map/map-mixins.md ## Base Map ::: danger This mixin is a manda...
3,How do I add a new layer to a KDK map?,4,0.6831,kdk/docs/api/map/utilities.md,markdown,Context: Utilities > Layers Source: kdk/docs/api/map/utilities.md ## Layers
4,How do I add a new layer to a KDK map?,5,0.6823,kdk/extras/tours/add-layer.js,javascript,"// kdk/extras/tours/add-layer.js module.exports = [{ target: '#import-layer', title: 'tours.add-layer.IMPORT_LAY..."
5,Où est implémentée la logique de géolocalisation dans KDK ?,1,0.6827,kdk/map/client/geolocation.js,javascript,"// kdk/map/client/geolocation.js import _ from 'lodash' import { Store, Events, utils, LocalStorage } from '../../co..."
6,Où est implémentée la logique de géolocalisation dans KDK ?,2,0.6763,kdk/map/client/i18n/map_fr.json,i18n_translations,"// kdk/map/client/i18n/map_fr.json :: errors {""errors"": {""GEOLOCATION_NOT_SUPPORTED"": ""La géolocalisation ne semble ..."
7,Où est implémentée la logique de géolocalisation dans KDK ?,3,0.6600,kdk/map/client/geocoder.js,javascript,// kdk/map/client/geocoder.js :: results }
8,Où est implémentée la logique de géolocalisation dans KDK ?,4,0.6589,kdk/docs/guides/basics/introduction.md,markdown,Context: Introduction to KDK > KDK internals Source: kdk/docs/guides/basics/introduction.md ## KDK internals Our m...
9,Où est implémentée la logique de géolocalisation dans KDK ?,5,0.6534,kdk/docs/api/map/composables/composables.location.md,markdown,Context: Location > Exposed Source: kdk/docs/api/map/composables/composables.location.md ## Exposed | Name | Type ...


## Gold-Set Retrieval Evaluation

This section replays the nb05 200-question gold set against the real Qdrant collection. It uses the same Qwen3 query prefix as nb05 and evaluates the returned chunks after de-duplicating them into file-level rankings. The chunk candidate window is intentionally wider than the final file cutoff because several chunks can come from the same source file.


In [7]:
GOLD_PATH = OUTPUT_DIR / "nb05_gold.json"
EVAL_PATH = OUTPUT_DIR / "nb06_qdrant_eval.json"
COMPARE_PATH = OUTPUT_DIR / "nb06_qdrant_vs_nb05.json"

queries = load_gold(GOLD_PATH)
query_en = [q.en for q in queries]
query_fr = [q.fr for q in queries]
print(f"[gold] queries={len(queries)}")


[gold] queries=200


In [8]:
client = create_qdrant_client(config.qdrant_url)
model = load_qwen_model()

t0 = time.perf_counter()
en_vectors = encode_queries(model, query_en)
fr_vectors = encode_queries(model, query_fr)

en_rankings = qdrant_file_rankings(client, config.collection_name, en_vectors, limit_chunks=500, max_files=20)
fr_rankings = qdrant_file_rankings(client, config.collection_name, fr_vectors, limit_chunks=500, max_files=20)
elapsed = time.perf_counter() - t0

rows = []
rows.extend(evaluate_file_rankings(en_rankings, queries, language="en", approach="qdrant_qwen3"))
rows.extend(evaluate_file_rankings(fr_rankings, queries, language="fr", approach="qdrant_qwen3"))
write_json(EVAL_PATH, rows)

del model, en_vectors, fr_vectors
gc.collect()

df_qdrant = pd.DataFrame(rows)
print(f"[eval] rows={len(df_qdrant)} seconds={elapsed:.1f} output={EVAL_PATH}")
df_qdrant.head()


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

[encode] queries=200 batch_size=16
[encode] queries=200 batch_size=16
[eval] rows=400 seconds=8.0 output=/home/felix/kalisio/knowledge/outputs/nb06_qdrant_eval.json


,approach,language,layer,query_id,is_negative,hit@k,recall@k,mrr,hit@1,recall@1,hit@5,recall@5,hit@10,recall@10
0,qdrant_qwen3,en,A_symbol,A-001,False,0,0.0,0.0,0,0.0,0,0.0,0,0.0
1,qdrant_qwen3,en,A_symbol,A-002,False,0,0.0,0.0,0,0.0,0,0.0,0,0.0
2,qdrant_qwen3,en,A_symbol,A-003,False,1,1.0,0.5,0,0.0,1,1.0,1,1.0
3,qdrant_qwen3,en,A_symbol,A-004,False,0,0.0,0.0,0,0.0,0,0.0,0,0.0
4,qdrant_qwen3,en,A_symbol,A-005,False,0,0.0,0.0,0,0.0,0,0.0,0,0.0


In [9]:
def summarize_eval(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.groupby(["approach", "language", "layer"])[["hit@1", "hit@5", "hit@10", "mrr"]]
        .mean(numeric_only=True)
        .round(3)
        .reset_index()
    )

qdrant_summary = summarize_eval(df_qdrant)
print(qdrant_summary.to_string(index=False))

positive_summary = (
    df_qdrant[~df_qdrant["is_negative"]]
    .groupby(["language"])[["hit@1", "hit@5", "hit@10", "mrr"]]
    .mean(numeric_only=True)
    .round(3)
)
print(positive_summary.to_string())


    approach language    layer  hit@1  hit@5  hit@10   mrr
qdrant_qwen3       en A_symbol  0.533  0.883   0.883 0.698
qdrant_qwen3       en   B_docs  0.288  0.725   0.862 0.463
qdrant_qwen3       en   C_code  0.289  0.733   0.889 0.448
qdrant_qwen3       en negative  0.467  0.000   0.000   NaN
qdrant_qwen3       fr A_symbol  0.483  0.883   0.883 0.661
qdrant_qwen3       fr   B_docs  0.288  0.688   0.800 0.460
qdrant_qwen3       fr   C_code  0.133  0.511   0.800 0.301
qdrant_qwen3       fr negative  0.600  0.000   0.000   NaN
          hit@1  hit@5  hit@10    mrr
language                             
en        0.368  0.778   0.876  0.536
fr        0.314  0.708   0.827  0.486


## Compare With nb05 Offline Qwen

The comparison below checks whether the real Qdrant pipeline stays close to nb05's in-memory Qwen evaluation. Small differences are acceptable because Qdrant uses an ANN index, but large differences would point to a prefix, normalization, distance, or payload-mapping issue.


In [10]:
NB05_RESULTS_PATH = OUTPUT_DIR / "nb05_results.json"
if NB05_RESULTS_PATH.exists():
    df_nb05 = pd.DataFrame(json.loads(NB05_RESULTS_PATH.read_text(encoding="utf-8")))
    df_nb05_qwen = df_nb05[df_nb05["approach"] == "qwen3-0.6b"].copy()
    df_nb05_qwen["approach"] = "nb05_memory_qwen3"
    compare_df = pd.concat([df_nb05_qwen, df_qdrant], ignore_index=True)
    compare_summary = summarize_eval(compare_df)
    write_json(COMPARE_PATH, compare_summary.to_dict(orient="records"))
    print(compare_summary.to_string(index=False))
    print(f"[compare] output={COMPARE_PATH}")
else:
    print(f"[compare] skipped: {NB05_RESULTS_PATH} not found")


         approach language    layer  hit@1  hit@5  hit@10   mrr
nb05_memory_qwen3       en A_symbol  0.500  0.883   0.883 0.684
nb05_memory_qwen3       en   B_docs  0.288  0.725   0.862 0.464
nb05_memory_qwen3       en   C_code  0.289  0.756   0.889 0.451
nb05_memory_qwen3       en negative  0.467  0.000   0.000   NaN
nb05_memory_qwen3       fr A_symbol  0.500  0.883   0.883 0.669
nb05_memory_qwen3       fr   B_docs  0.288  0.688   0.800 0.465
nb05_memory_qwen3       fr   C_code  0.156  0.511   0.800 0.317
nb05_memory_qwen3       fr negative  0.600  0.000   0.000   NaN
     qdrant_qwen3       en A_symbol  0.533  0.883   0.883 0.698
     qdrant_qwen3       en   B_docs  0.288  0.725   0.862 0.463
     qdrant_qwen3       en   C_code  0.289  0.733   0.889 0.448
     qdrant_qwen3       en negative  0.467  0.000   0.000   NaN
     qdrant_qwen3       fr A_symbol  0.483  0.883   0.883 0.661
     qdrant_qwen3       fr   B_docs  0.288  0.688   0.800 0.460
     qdrant_qwen3       fr   C_code  0.1

## Optional Vector Store Comparison

This section compares Chroma and LanceDB against Qdrant using the same `vector_records` generated for the Qdrant build. Install the optional dependencies first if the cell reports that they are missing.


In [11]:
import importlib.util

optional_backends = {
    "chromadb": importlib.util.find_spec("chromadb") is not None,
    "lancedb": importlib.util.find_spec("lancedb") is not None,
}
print(optional_backends)
if not all(optional_backends.values()):
    print("Install missing optional stores with: python -m pip install chromadb lancedb")


{'chromadb': True, 'lancedb': True}


In [12]:
if "vector_records" not in globals():
    raise RuntimeError("Run the Qdrant build cell first so vector_records are available.")

client = create_qdrant_client(config.qdrant_url)
model = load_qwen_model()
en_vectors = encode_queries(model, query_en)
fr_vectors = encode_queries(model, query_fr)

store_rows = []
store_timings = []

t0 = time.perf_counter()
en_rankings = qdrant_file_rankings(client, config.collection_name, en_vectors, limit_chunks=500, max_files=20)
fr_rankings = qdrant_file_rankings(client, config.collection_name, fr_vectors, limit_chunks=500, max_files=20)
store_timings.append({"store": "qdrant", "seconds": round(time.perf_counter() - t0, 3)})
store_rows.extend(evaluate_file_rankings(en_rankings, queries, language="en", approach="qdrant_qwen3"))
store_rows.extend(evaluate_file_rankings(fr_rankings, queries, language="fr", approach="qdrant_qwen3"))

if optional_backends.get("chromadb"):
    t0 = time.perf_counter()
    chroma_collection = build_chroma_collection(
        vector_records,
        path=store_config.chroma_path,
        collection_name=store_config.chroma_collection_name,
        recreate=True,
    )
    build_seconds = time.perf_counter() - t0
    t0 = time.perf_counter()
    en_rankings = chroma_file_rankings(chroma_collection, en_vectors, limit_chunks=500, max_files=20)
    fr_rankings = chroma_file_rankings(chroma_collection, fr_vectors, limit_chunks=500, max_files=20)
    store_timings.append({"store": "chroma", "build_seconds": round(build_seconds, 3), "query_seconds": round(time.perf_counter() - t0, 3)})
    store_rows.extend(evaluate_file_rankings(en_rankings, queries, language="en", approach="chroma_qwen3"))
    store_rows.extend(evaluate_file_rankings(fr_rankings, queries, language="fr", approach="chroma_qwen3"))

if optional_backends.get("lancedb"):
    t0 = time.perf_counter()
    lance_table = build_lancedb_table(
        vector_records,
        path=store_config.lancedb_path,
        table_name=store_config.lancedb_table_name,
        mode="overwrite",
    )
    build_seconds = time.perf_counter() - t0
    t0 = time.perf_counter()
    en_rankings = lancedb_file_rankings(lance_table, en_vectors, limit_chunks=500, max_files=20)
    fr_rankings = lancedb_file_rankings(lance_table, fr_vectors, limit_chunks=500, max_files=20)
    store_timings.append({"store": "lancedb", "build_seconds": round(build_seconds, 3), "query_seconds": round(time.perf_counter() - t0, 3)})
    store_rows.extend(evaluate_file_rankings(en_rankings, queries, language="en", approach="lancedb_qwen3"))
    store_rows.extend(evaluate_file_rankings(fr_rankings, queries, language="fr", approach="lancedb_qwen3"))

del model, en_vectors, fr_vectors
gc.collect()

df_store_compare = pd.DataFrame(store_rows)
write_json(OUTPUT_DIR / "nb06_vector_store_compare.json", store_rows)
write_json(OUTPUT_DIR / "nb06_vector_store_timings.json", store_timings)
print(pd.DataFrame(store_timings).to_string(index=False))
print(summarize_eval(df_store_compare).to_string(index=False))


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

[encode] queries=200 batch_size=16
[encode] queries=200 batch_size=16


[2026-04-27T14:31:15Z WARN  lance::dataset::write::insert] No existing dataset at /home/felix/kalisio/knowledge/notebooks/vector_db/lancedb_nb06/kalisio_qwen3_nb06_v1.lance, it will be created


  store  seconds  build_seconds  query_seconds
 qdrant    6.678            NaN            NaN
 chroma      NaN          8.083          7.993
lancedb      NaN          0.991         55.117
     approach language    layer  hit@1  hit@5  hit@10   mrr
 chroma_qwen3       en A_symbol  0.533  0.883   0.883 0.698
 chroma_qwen3       en   B_docs  0.288  0.725   0.862 0.463
 chroma_qwen3       en   C_code  0.289  0.733   0.889 0.448
 chroma_qwen3       en negative  0.467  0.000   0.000   NaN
 chroma_qwen3       fr A_symbol  0.483  0.883   0.883 0.661
 chroma_qwen3       fr   B_docs  0.288  0.688   0.800 0.460
 chroma_qwen3       fr   C_code  0.133  0.511   0.800 0.301
 chroma_qwen3       fr negative  0.600  0.000   0.000   NaN
lancedb_qwen3       en A_symbol  0.533  0.883   0.883 0.698
lancedb_qwen3       en   B_docs  0.300  0.725   0.862 0.470
lancedb_qwen3       en   C_code  0.267  0.733   0.889 0.437
lancedb_qwen3       en negative  0.533  0.000   0.000   NaN
lancedb_qwen3       fr A_symbol 

## Vector Store Summary

This final summary separates positive retrieval quality from negative-query safety. The vector store should be chosen only after checking both dimensions: a store can match retrieval quality while still leaving the negative-question problem untouched.


In [13]:
STORE_COMPARE_PATH = OUTPUT_DIR / "nb06_vector_store_compare.json"
STORE_TIMINGS_PATH = OUTPUT_DIR / "nb06_vector_store_timings.json"

store_compare = pd.DataFrame(json.loads(STORE_COMPARE_PATH.read_text()))
store_timings = pd.DataFrame(json.loads(STORE_TIMINGS_PATH.read_text()))

store_compare["store"] = store_compare["approach"].str.replace("_qwen3", "", regex=False)
positive = store_compare[~store_compare["is_negative"]].copy()
negative = store_compare[store_compare["is_negative"]].copy()

positive_summary = (
    positive.groupby("store")
    .agg(
        positive_rows=("query_id", "count"),
        hit_at_1=("hit@1", "mean"),
        hit_at_5=("hit@5", "mean"),
        hit_at_10=("hit@10", "mean"),
        recall_at_5=("recall@5", "mean"),
        mrr=("mrr", "mean"),
    )
    .reset_index()
)

negative_summary = (
    negative.groupby("store")
    .agg(
        negative_rows=("query_id", "count"),
        negative_safe_at_1=("hit@1", "mean"),
        negative_safe_at_5=("hit@5", "mean"),
        negative_safe_at_10=("hit@10", "mean"),
    )
    .reset_index()
)

timing_summary = store_timings.copy()
timing_summary["build_seconds"] = timing_summary.get("build_seconds", pd.Series(index=timing_summary.index, dtype="float64")).fillna(0.0)
timing_summary["query_seconds"] = timing_summary.get("query_seconds", pd.Series(index=timing_summary.index, dtype="float64")).fillna(timing_summary.get("seconds"))
timing_summary = timing_summary[["store", "build_seconds", "query_seconds"]]

store_summary = (
    positive_summary
    .merge(negative_summary, on="store", how="outer")
    .merge(timing_summary, on="store", how="left")
    .sort_values(["hit_at_5", "mrr", "query_seconds"], ascending=[False, False, True])
)

qdrant_row = store_summary.loc[store_summary["store"] == "qdrant"].iloc[0]
store_summary["delta_hit_at_5_vs_qdrant"] = store_summary["hit_at_5"] - qdrant_row["hit_at_5"]
store_summary["delta_mrr_vs_qdrant"] = store_summary["mrr"] - qdrant_row["mrr"]

language_summary = (
    positive.groupby(["store", "language"])
    .agg(hit_at_5=("hit@5", "mean"), hit_at_10=("hit@10", "mean"), mrr=("mrr", "mean"))
    .reset_index()
    .sort_values(["store", "language"])
)

percent_cols = [
    "hit_at_1", "hit_at_5", "hit_at_10", "recall_at_5", "mrr",
    "negative_safe_at_1", "negative_safe_at_5", "negative_safe_at_10",
    "delta_hit_at_5_vs_qdrant", "delta_mrr_vs_qdrant",
]
summary_display = store_summary.copy()
for col in percent_cols:
    summary_display[col] = summary_display[col].map(lambda value: "" if pd.isna(value) else f"{value:.1%}")
for col in ["build_seconds", "query_seconds"]:
    summary_display[col] = summary_display[col].map(lambda value: "" if pd.isna(value) else f"{value:.3f}s")

language_display = language_summary.copy()
for col in ["hit_at_5", "hit_at_10", "mrr"]:
    language_display[col] = language_display[col].map(lambda value: f"{value:.1%}")

print("Main comparison")
print(summary_display.to_string(index=False))
print("\nPositive queries by language")
print(language_display.to_string(index=False))

max_delta = store_summary[["delta_hit_at_5_vs_qdrant", "delta_mrr_vs_qdrant"]].abs().max().max()
best_query = store_summary.sort_values("query_seconds").iloc[0]
slowest_query = store_summary.sort_values("query_seconds").iloc[-1]

print("\nSummary")
if max_delta < 0.005:
    print("- Retrieval quality is effectively identical across Qdrant, Chroma, and LanceDB on this gold set.")
else:
    best_quality = store_summary.sort_values(["hit_at_5", "mrr"], ascending=False).iloc[0]
    print(f"- {best_quality['store']} has the best retrieval score, but check whether the margin is meaningful before changing stores.")
print(f"- Query time is best for {best_query['store']} in this run and slowest for {slowest_query['store']}.")
print("- Negative-query safety remains weak for every store at @5/@10, so this should be solved in the retrieval policy or reranking/abstention layer, not by switching vector databases.")


Main comparison
  store  positive_rows hit_at_1 hit_at_5 hit_at_10 recall_at_5   mrr  negative_rows negative_safe_at_1 negative_safe_at_5 negative_safe_at_10 build_seconds query_seconds delta_hit_at_5_vs_qdrant delta_mrr_vs_qdrant
lancedb            370    34.1%    74.6%     85.1%       59.0% 51.1%             30              56.7%               0.0%                0.0%        0.991s       55.117s                     0.3%                0.0%
 qdrant            370    34.1%    74.3%     85.1%       58.9% 51.1%             30              53.3%               0.0%                0.0%        0.000s        6.678s                     0.0%                0.0%
 chroma            370    34.1%    74.3%     85.1%       58.9% 51.1%             30              53.3%               0.0%                0.0%        8.083s        7.993s                     0.0%                0.0%

Positive queries by language
  store language hit_at_5 hit_at_10   mrr
 chroma       en    77.8%     87.6% 53.6%
 chroma   

## Decision

If Qdrant Qwen3 stays close to nb05 memory Qwen3, the retrieval base is ready for the next Phase 4 step: abstention and negative-question handling. If the metrics diverge, inspect query prefixing, vector normalization, Qdrant distance configuration, and file-level de-duplication before moving on.
